# Rocket Model Analysis

In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [2]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists())
sys.path.insert(0, str(root))

In [16]:
"""
ALL SYMBOLS, CONSTANTS AND FUNCTIONS
"""
# Variables
t = sp.Symbol("t")
# Adjustable parameters
k1, k2, T = sp.symbols("k1 k2 T")
# Constants
ka = 0.0655         # actuator's push gain
tm = 0.9370314843   # engine bandwidth (1/tm = 1.0672)
c0 = 9.81           # gravitational coupling
mu = 1.20           # aerodynamic rigidity
ad = 0.35           # aerodynamic damping
kd = 8.5            # nozzle's effectivity
ke = 0.12           # coupling from missalignment
ts = 0.05           # servo's bandwidth (1/ts = 20)
lmd = 0.6           # IMO's speed-position ratio
# Inputs
u1 = sp.Function('u1')(t)
u2 = sp.Function('u2')(t)
inputs = [u1, u2]
# States
acc = sp.Function('a')(t)   # axial acceleration
theta = sp.Function('0')(t) # angular velocity
omega = sp.Function('w')(t) # momentum balacing
delta = sp.Function('d')(t) # nozzle's servo dynamic
states = [acc, theta, omega, delta]
# Outputs
y1 = sp.Function('y1')(t)
y2 = sp.Function('y2')(t)
outputs = [y1, y2]

"""
EQUATION SYSTEM DEFINITION
"""
# States equation system
dx1 = sp.Eq(acc.diff(t), -((1 + k1*ka)/tm)*acc - c0*theta + ((k1*ka)/tm)*u1)
dx2 = sp.Eq(theta.diff(t), omega)
dx3 = sp.Eq(omega.diff(t), mu*theta - ad*omega + kd*delta + ke*acc)
dx4 = sp.Eq(delta.diff(t), -(1/ts)*delta -(k2/ts)*theta - ((lmd*k2)/ts)*omega + (k2/ts)*u2)
state_system = [dx1, dx2, dx3, dx4]
# Output equation system
out1 = sp.Eq(y1, acc)
out2 = sp.Eq(y2, theta)
output_system = [out1, out2]

## (A) Modelado del Sistema

1) Las entradas son los valores que deseamos de aceleración axial y ángulo de cabeceo para nuestro cohete por lo que serán las variables por comparar para obtener nuestro error.
Los bloques A/D son convertidores de señales analógicas a digitales o ADC para que la computadora pueda interpretar los datos que recibe de los sensores.
Computadora, obtiene las entradas y la retroalimentación y las compara para obtener nuestro error el cual se multiplicará por las constantes K1 y K2 para la respectiva entrada, actualizará el motor y el servo con los resultados en un tiempo dado que será el tiempo de muestreo. 

- D/A: Convertidor digital a analógico para poder comunicarse con el motor y el
servo.
- Amplificadores: Para que la señal digital débil se amplifique e influya en el
comportamiento del motor y el servo.
- Dinámica del cohete, los valores de aceleración y de nuestras entradas
afectarán a la dinámica de nuestro cohete, afectando su aceleración axial y su
ángulo de cabeceo.
- k_a: Ganancia del actuador de empuje, constante de proporcionalidad que transforma
la señal de control en una aceleración axial física.
- 1/T_m: Ancho de banda del motor, inercia del motor como un retardo de primer orden
indica qué tan rápido puede cambiar el empuje físico ante un cambio de comando.
- c_theta: Acoplamiento gravitatorio, es la pérdida de aceleración axial por la inclinación del
cohete.
- μ: Rigidez aerodinámica (desestabilizante), es el momento de giro generado por el
viento sobre el cohete cuando el CP está sobre el CM.
- a_d: Amortiguamiento aerodinámico, es la resistencia del aire ante la rotación del
cohete.
- k_delta: Efectividad de la tobera, es la aceleración angular que se produce por radián de
deflexión de la tobera.
- k_epsilon: Acoplamiento por desalineación del empuje, al no pasar el vector del empuje por
el CM genera un torque parásito que perturba la aceleración angular.
- 1/T_S: Ancho de banda del servo, es la respuesta del servo ante un comando, es más
rápida su respuesta que la del motor.
- λ: Proporción velocidad / Posición de la IMU, pondera la señal de velocidad angular
en relación con el ángulo en la lectura, introduce un amortiguamiento artificial (como
un control PD).
- K1: Ganancia del canal de empuje, modula la retroalimentación de la aceleración
axial, el polo efectivo se desplaza a la izquierda más rápido acelerando la respuesta
del empuje y reduciendo el error en régimen permanente.
El error en régimen permanente (también llamado error en estado estacionario) es la diferencia que queda entre el valor de la señal de referencia (lo que queremos que haga el sistema) y el valor real de la salida cuando el tiempo pasa y el sistema ya se ha
estabilizado.
- K2: Ganancia del canal de dirección, determina la intensidad con la que se mueve la
tobera para corregir la actitud del cohete, busca estabilizar la inestabilidad
aerodinámica natural del cohete, si es muy baja o cero no habrá suficiente torque en
la tobera y el cohete seguirá inestable por las fuerzas que influyen en él.
- T: Período de muestreo, tiempo entre un ciclo y el siguiente, es el tiempo en donde se leerá, calculará y enviarán las órdenes al sistema. El muestreo ocurre en la computadora donde: Lee, calcula, manda la orden nueva y espera en el tiempo de muestreo especificado.

2) Las trayectorias que se eliminan cuando K2=0 son las que conforman a la ecuación del cambio del ángulo entre el eje del cohete y la tobera, provocando que la tobera siempre regrese a una posición δ(t)=0 de forma progresiva sin importar la inclinación del cohete. No podríamos controlarlo con nuestras entradas. Si no hubiera acoplamiento cruzado en la aceleración axial no se tomaría en cuenta la pérdida de aceleración por la inclinación del cohete y en la velocidad angular no se tomaría en cuenta la influencia de la aceleración por ese torque parásito por la desviación de la fuerza de empuje con el centro de masa.

3) Ecuación de Estados
- a(t) es la aceleración axial de nuestro cohete.
- θ(t) es el ángulo de cabeceo de nuestro cohete.
- w(t) es la velocidad angular del cabeceo (velocidad con la que el cohete rota).
- δ(t) es el ángulo de deflexión de la tobera.

Sí eliminamos x4 entonces asumimos que el servomotor de la tobera responde automáticamente a los comandos enviados por la computadora, lo que aparentaría que el cohete fuera más fácil de controlar de lo que realmente es ya que ignoramos desfases e inestabilidad que pueden producirse por el ángulo de deflexión de la tobera.

ECUACIÓN DE ESTADOS SIMBÓLICA:
```
┌      ┐   ┌                                                  ┐ ┌      ┐   ┌                ┐
│ ȧ(t) │   │ -(1 + K1*Ka)/Tm    -c_θ         0         0      │ │ a(t) │   │ K1*Ka/Tm   0   │ ┌       ┐
│ θ̇(t) │   │        0             0          1         0      │ │ θ(t) │   │    0       0   │ │ u1(t) │
│ ẇ(t) │ = │       K_ε            μ        -a_d       K_δ     │ │ w(t) │ + │    0       0   │ │ u2(t) │
│ δ̇(t) │   │        0          -K2/Ts    -λ*K2/Ts    -1/Ts    │ │ δ(t) │   │    0     K2/Ts │ └       ┘
└      ┘   └                                                  ┘ └      ┘   └                ┘
```
ECUACIÓN DE SALIDA:
```
┌       ┐   ┌            ┐ ┌      ┐
│ y1(t) │ = │ 1  0  0  0 │ │ a(t) │
│ y2(t) │   │ 0  1  0  0 │ │ θ(t) │
└       ┘   └            ┘ └      ┘
```

In [17]:
from src.system_rep import get_all_matrixes, display_state_equations

# Extracción de matrices a partir del sistema de ecuaciones
A, B, C, D = get_all_matrixes(state_system, output_system, states, inputs, outputs)
print("A = ", A)
print("B = ", B)
print("C = ", C)
print("D = ", D)
display_state_equations(A, B, C, D, states, inputs)

A =  [[-0.0699015999968572*k1 - 1.06719999995202, -9.81000000000000, 0, 0], [0, 0, 1, 0], [0.120000000000000, 1.20000000000000, -0.350000000000000, 8.50000000000000], [0, -20.0*k2, -12.0*k2, -20.0000000000000]]
B =  [[0.0699015999968572*k1, 0], [0, 0], [0, 0], [0, 20.0*k2]]
C =  [[1, 0, 0, 0], [0, 1, 0, 0]]
D =  [[0, 0], [0, 0]]
STATE EQUATIONS
------------------------------------------------------------
dx1/dt = (-0.0699015999968572*k1 - 1.06719999995202)*a(t) - (9.81000000000000)*0(t) + (0.0699015999968572*k1)*u1(t)
dx2/dt = (1)*w(t)
dx3/dt = (0.120000000000000)*a(t) + (1.20000000000000)*0(t) - (0.350000000000000)*w(t) + (8.50000000000000)*d(t)
dx4/dt = (-20.0*k2)*0(t) - (12.0*k2)*w(t) - (20.0000000000000)*d(t) + (20.0*k2)*u2(t)

OUTPUT EQUATIONS
------------------------------------------------------------
y1 = (1)*a(t)
y2 = (1)*0(t)


In [30]:
p = sp.Symbol("p", real=True)
matA = sp.Matrix(A)
matB = sp.Matrix(B)
matC = sp.Matrix(C)

n = matA.shape
pI_A = p * sp.eye(4) - matA
invA = (pI_A).inv()
G = matC * invA * matB
G = G.applyfunc(sp.simplify)

for i in range(G.shape[0]):
    for j in range(G.shape[1]):
        num, den = sp.fraction(sp.cancel(G[i, j]))
        print(f"G_{i+1},{j+1}:")
        print(f"Numerador: {num}")
        print(f"Denominador: {den}")

G_1,1:
Numerador: 7.12996319967944*k1*k2*p + 11.8832719994657*k1*k2 + 0.0699015999968572*k1*p**3 + 1.42249755993604*k1*p**2 + 0.405429279981772*k1*p - 1.67763839992457*k1
Denominador: 7.12996319967944*k1*k2*p + 11.8832719994657*k1*k2 + 0.0699015999968572*k1*p**3 + 1.42249755993604*k1*p**2 + 0.405429279981772*k1*p - 1.67763839992457*k1 + 102.0*k2*p**2 + 278.854399995106*k2*p + 181.423999991843*k2 + 1.0*p**4 + 21.417199999952*p**3 + 27.5175199990236*p**2 - 16.6330400002783*p - 2.06879999884844
G_1,2:
Numerador: -1667.7*k2
Denominador: 7.12996319967944*k1*k2*p + 11.8832719994657*k1*k2 + 0.0699015999968572*k1*p**3 + 1.42249755993604*k1*p**2 + 0.405429279981772*k1*p - 1.67763839992457*k1 + 102.0*k2*p**2 + 278.854399995106*k2*p + 181.423999991843*k2 + 1.0*p**4 + 21.417199999952*p**3 + 27.5175199990236*p**2 - 16.6330400002783*p - 2.06879999884844
G_2,1:
Numerador: 0.00838819199962287*k1*p + 0.167763839992457*k1
Denominador: 7.12996319967944*k1*k2*p + 11.8832719994657*k1*k2 + 0.069901599996857

5) Ecuación de Salida

Casos:
- Se dispone únicamente de un acelerómetro.
- Se dispone únicamente de la medición de ángulo de cabeceo.
- Se desea además registrar la velocidad angular w para monitorear cargas estructurales.

Si solo tenemos un acelerómetro:
```
y(t) = [1, 0, 0, 0] [a(t), θ(t), w(t), δ(t)]ᵀ
y(t) = a(t)
```

Si solo podemos medir el ángulo de cabeceo:
```
y(t) = [0, 1, 0, 0] [a(t), θ(t), w(t), δ(t)]ᵀ
y(t) = θ(t)
```

Si también deseamos registrar la velocidad angular:
```
y(t) = [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0]] [a(t), θ(t), w(t), δ(t)]ᵀ
y1(t) = a(t)
y2(t) = θ(t)
y3(t) = w(t)
```

## (B) Estabilidad y Respuesta temporal

In [ ]:
# Inputs definition
def f1(t):
    return np.where(t >= 1.0, 2.0, 0)
def f2(t):
    return np.sin(0.5 * t)

T = np.linspace(0.0, 20.0, 1000)
U = np.column_stack((f1(T), f2(T)))

# Output calculation
state_space = signal.StateSpace(A, B, C, D)
t_out, y_out, x_out = signal.lsim(state_space, U=U, T=T)

In [ ]:
plt.figure(figsize=(8, 4))
# plt.plot(t_out, y_out[:,0], label="y1")
# plt.plot(t_out, y_out[:,1], label="y2")
plt.plot(t_out, x_out[:,0], label="x1")
plt.plot(t_out, x_out[:,1], label="x2")
plt.plot(t_out, x_out[:,2], label="x3")
plt.plot(t_out, x_out[:,3], label="x4")
plt.xlabel("time (s)")
plt.ylabel("States/output")
plt.title("System Response")
plt.legend()
plt.grid(True)
plt.show()

## (C) Respuesta en frecuencia

## (D) Observabilidad y Controlabilidad

## (E) Seleccion de ganancias y muestreo